In [89]:
import pickle
import pandas as pd
import re
import requests

In [90]:
DEFAULT_NO_IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/1/14/No_Image_Available.jpg"

In [91]:
def image_preprocessor(s):
    if pd.isna(s) or not isinstance(s, str):
        return [DEFAULT_NO_IMAGE_URL]
    
    urls = re.findall(r'"(https?://[^"]+)"', s)
    
    if not urls:
        return [DEFAULT_NO_IMAGE_URL]
        
    return urls

In [92]:
class ImageCollection:
    def __init__(self, image_dataframe):
        self.collection = image_dataframe.set_index('RecipeId')
        
    def get_urls(self, recipe_id):
        try:
            result = self.collection.loc[recipe_id, 'Images']
            
            if isinstance(result, str):
                return [result]
            
            return result.tolist()
        except KeyError:
            return []

In [93]:
with open('resource/recipes.pkl', 'rb') as f:
    recipes = pickle.load(f)

In [94]:
recipes['Image_List'] = recipes['Images'].apply(image_preprocessor)

image_df = recipes[['RecipeId', 'Image_List']].copy()

image_df = image_df.explode('Image_List')
image_df = image_df.dropna(subset=['Image_List']).reset_index(drop=True)
image_df = image_df.rename(columns={'Image_List': 'Images'})

image_df['RecipeId'] = pd.to_numeric(image_df['RecipeId'], errors='coerce')

In [95]:
image_collector = ImageCollection(image_df)

In [112]:
image_collector.get_urls(38)

['https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/YUeirxMLQaeE1h3v3qnM_229%20berry%20blue%20frzn%20dess.jpg',
 'https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/AFPDDHATWzQ0b1CDpDAT_255%20berry%20blue%20frzn%20dess.jpg',
 'https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/UYgf9nwMT2SGGJCuzILO_228%20berry%20blue%20frzn%20dess.jpg',
 'https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/PeBMJN2TGSaYks2759BA_20140722_202142.jpg',
 'https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/picuaETeN.jpg',
 'https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/pictzvxW5.jpg']

In [113]:
with open('resource/image_collection.pkl', 'wb') as f:
    pickle.dump(image_collector, f)